In [1]:
import random
import numpy as np
from math import pi
import statistics
import json
import os

In [ ]:
def generate_rectangle(area):
    a = random.randint(1, int(area**0.5))
    b = area // a
    x1 = random.randint(0, 50000 - a)
    y1 = random.randint(0, 25000 - b)
    vertices = [
        {"x": x1, "y": y1},
        {"x": x1 + a, "y": y1},
        {"x": x1 + a, "y": y1 + b},
        {"x": x1, "y": y1 + b},
        {"x": x1, "y": y1},
    ]
    new_area = a * b
    return vertices, new_area

In [ ]:
def generate_cell_rectangle(a, b):
    x1 = random.randint(0, 50000 - a)
    y1 = random.randint(0, 25000 - b)
    vertices = [
        {"x": x1, "y": y1},
        {"x": x1 + a, "y": y1},
        {"x": x1 + a, "y": y1 + b},
        {"x": x1, "y": y1 + b},
        {"x": x1, "y": y1},
    ]
    return vertices

In [ ]:
def create_cell(cell_class_ind):
    cell_classes = {1: "клетка ЩЖ (норма)",
                    2: "клетка ЩЖ Гюртле",
                    3: "клетка ЩЖ с несколькими ядрами",
                    4: "клетка ЩЖ с псевдовключением",
                    5: "клетка ЩЖ с бороздой в ядре",
                    6: "лимфоцит"
                   }
    cell = {}
    cell["cell_class"] = cell_classes[cell_class_ind]
    
    rand_probs = np.random.random(6)
    rand_probs = [round(n / rand_probs.sum(), 2) for n in rand_probs]
    rand_probs = sorted(rand_probs, reverse=True)
    cell["probs"] = rand_probs
    to_change = cell["probs"][cell_class_ind - 1]
    cell["probs"][cell_class_ind - 1] = cell["probs"][0]
    cell["probs"][0] = to_change

    if cell_class_ind == 6:
        min_d = 5
        max_d = 10
    elif cell_class_ind == 2:
        min_d = 20
        max_d = 40
    else:
        min_d = 10
        max_d = 20
    diameters = sorted([random.randint(min_d, max_d) for i in range(2)], reverse=True)
    cell["min_d"] = diameters[1]
    cell["max_d"] = diameters[0]
    
    radius_min = diameters[1] / 2
    radius_max = diameters[0] / 2
    cell["area"] = round(pi * radius_min * radius_max, 2) 
    cell["aspect_ratio"] = round(cell["min_d"] / cell["max_d"], 2)
    
    
    if cell_class_ind == 6:
        min_n_c_r = 0.99
        max_n_c_r = 1
    elif cell_class_ind == 2:
        min_n_c_r = 0.1
        max_n_c_r = 0.6
    else:
        min_n_c_r = 0.6
        max_n_c_r = 1
    cell["nuclear_cytoplasmic_ratio"] = round(random.uniform(min_n_c_r, max_n_c_r), 2)
    
    perimeter = 2 * pi * (((radius_min ** 2 + radius_max ** 2) / 2) ** 0.5)
    cell["circularity"] = round((4 * pi * cell["area"]) / (perimeter * perimeter), 2)
    
    cell["points"] = generate_cell_rectangle(diameters[0], diameters[1])
    
    return cell

In [ ]:
def create_cluster(cluster_class_ind, cluster_type):
    cluster_classes = {1: "бесформенная структура с упорядоченным расположением клеток",
                    2: "бесформенная структура с неупорядоченным расположением клеток",
                    3: "микрофолликулярная структура",
                    4: "трабекулярная структура",
                    5: "папиллярная структура"
                   }
    cluster = {}
    cluster["cluster_class"] = cluster_classes[cluster_class_ind]
    
    rand_probs = np.random.random(5)
    rand_probs = [round(n / rand_probs.sum(), 2) for n in rand_probs]
    rand_probs = sorted(rand_probs, reverse=True)
    cluster["probs"] = rand_probs
    to_change = cluster["probs"][cluster_class_ind - 1]
    cluster["probs"][cluster_class_ind - 1] = cluster["probs"][0]
    cluster["probs"][0] = to_change
    
    if cluster_type == 'usual':
        min_th_cell_num = 10
        max_th_cell_num = 50
    elif cluster_type == 'small':
        min_th_cell_num = 10
        max_th_cell_num = 12
    cluster["th_cell_num"] = random.randint(min_th_cell_num, max_th_cell_num)
    initial_area = cluster["th_cell_num"] * 500
    cluster["points"], cluster["area"] = generate_rectangle(initial_area)
    
    return cluster

In [ ]:
def generate_analysis_output(wsi_id, wsi_class_ind, cell_numbers, clusters_numbers, cluster_type, json_path):
    data = {}
    
    wsi_classes = {0: "Bethesda 1",
                   1: "Bethesda 2",
                   2: "Bethesda 3",
                   3: "Bethesda 4",
                   4: "Bethesda 5",
                   5: "Bethesda 6"}
    data["wsi_id"] = wsi_id
    data["wsi_class"] = wsi_classes[wsi_class_ind]
    rand_probs = np.random.random(6)
    rand_probs = [round(n / rand_probs.sum(), 2) for n in rand_probs]
    rand_probs = sorted(rand_probs, reverse=True)
    data["probs"] = rand_probs
    to_change = data["probs"][wsi_class_ind]
    data["probs"][wsi_class_ind] = data["probs"][0]
    data["probs"][0] = to_change
    
    cluster_characteristics = {}
    total_cluster_areas = []
    total_clusters = []
    total_th_cell_num = []
    
    cell_characteristics = {}
    cell_characteristics["cellularity"] = 0
    total_cells = []
    total_areas = []
    total_diameters = []
    total_aspect_ratios = []
    total_nuclear_cytoplasmic_ratio = []
    total_circularity = []
    
    cluster_counter = 0
    for key in cluster_numbers:
        for i in range(cluster_numbers[key]):
            cluster = create_cluster(key, cluster_type)
            cluster["cluster_id"] = cluster_counter
            cluster_counter += 1
            total_clusters.append(cluster)
            total_cluster_areas.append(cluster["area"])
            total_th_cell_num.append(cluster["th_cell_num"])
    
    cluster_characteristics["ordered_cells_shapeless_cluster_num"] = cluster_numbers[1]
    cluster_characteristics["disordered_cells_shapeless_cluster_num"] = cluster_numbers[2]
    cluster_characteristics["microfollicle_num"] = cluster_numbers[3]
    cluster_characteristics["trabecula_num"] = cluster_numbers[4]
    cluster_characteristics["papillary_num"] = cluster_numbers[5]
    
    cluster_characteristics["mean_cluster_area"] = round(statistics.mean(total_cluster_areas), 2)
    cluster_characteristics["mean_th_cell_num_in_clusters"] = round(statistics.mean(total_th_cell_num), 2)
    
    input_cells_num = 0
    for key in cell_numbers:
        if key != 6:
            input_cells_num += cell_numbers[key]
            
    if input_cells_num < sum(total_th_cell_num):
        if cluster_type == 'usual':
            min_add_cells = 10
            max_add_cells = 50
        elif cluster_type == 'small':
            min_add_cells = 1
            max_add_cells = 3
        cell_numbers[1] += sum(total_th_cell_num) - input_cells_num + random.randint(min_add_cells, max_add_cells)
            
    counter = 0
    for key in cell_numbers:
        if key != 6:
            cell_characteristics["cellularity"] += cell_numbers[key]
        for i in range(cell_numbers[key]):
            cell = create_cell(key)
            cell["cell_id"] = counter
            counter += 1
            total_cells.append(cell)
            total_areas.append(cell["area"])
            total_diameters.append(cell["min_d"])
            total_diameters.append(cell["max_d"])
            total_aspect_ratios.append(cell["aspect_ratio"])
            total_nuclear_cytoplasmic_ratio.append(cell["nuclear_cytoplasmic_ratio"])
            total_circularity.append(cell["circularity"])
                
    cell_characteristics["th_norm_cell_num"] = cell_numbers[1]
    cell_characteristics["th_gurtle_cell_num"] = cell_numbers[2]
    cell_characteristics["th_multiple_nuclei_cell_num"] = cell_numbers[3]
    cell_characteristics["th_pseudoinclusion_cell_num"] = cell_numbers[4]
    cell_characteristics["th_groove_cell_num"] = cell_numbers[5]
    cell_characteristics["lymphocyte_num"] = cell_numbers[6]
    
    cell_characteristics["mean_th_cell_area"] = round(statistics.mean(total_areas), 2)
    cell_characteristics["mean_th_cell_diameter"] = round(statistics.mean(total_diameters), 2)
    cell_characteristics["mean_th_cell_aspect_ratio"] = round(statistics.mean(total_aspect_ratios), 2)
    cell_characteristics["mean_th_cell_nuclear_cytoplasmic_ratio"] = round(statistics.mean(total_nuclear_cytoplasmic_ratio), 2)
    cell_characteristics["mean_th_cell_circularity"] = round(statistics.mean(total_circularity), 2)

    data["cluster_characteristics"] = cluster_characteristics
    data["cell_clusters"] = total_clusters
    data["cell_characteristics"] = cell_characteristics
    data["cells"] = total_cells
    
    with open(json_path, 'w') as f:
        json.dump(data, f, ensure_ascii=False)

In [ ]:
# Индексы и классы снимков обычных (wsi_class_ind): 
# 0 - Bethesda 1 
# 1 - Bethesda 2 
# 2 - Bethesda 3 
# 3 - Bethesda 4 
# 4 - Bethesda 5 
# 5 - Bethesda 6

# Индексы и классы скоплений клеток (cluster_numbers): 
# 0 - неинформативно 
# 1 - бесформенная структура с упорядоченным расположением клеток 
# 2 - бесформенная структура с неупорядоченным расположением клеток 
# 3 - микрофолликулярная структура 
# 4 - трабекулярная структура 
# 5 - папиллярная структура

# Индексы и классы клеток (cell_numbers):
# 0 - неинформативно
# 1 - клетка ЩЖ (норма)
# 2 - клетка ЩЖ Гюртле
# 3 - клетка ЩЖ с несколькими ядрами
# 4 - клетка ЩЖ с псевдовключением
# 5 - клетка ЩЖ с бороздой в ядре
# 6 - лимфоцит

In [ ]:
# задать параметры в соответствии с указанными выше обозначениями
wsi_id = 4
wsi_class_ind = 2 # нумерация с 0
cluster_numbers = {1: 3,
                2: 3,
                3: 0,
                4: 0,
                5: 0
                  }
cell_numbers = {1: 2,
                2: 20,
                3: 0,
                4: 0,
                5: 0,
                6: 30
               }
cluster_type = 'small' # usual or small
json_path = '4.json'

generate_analysis_output(wsi_id, wsi_class_ind, cell_numbers, cluster_numbers, cluster_type, json_path)

# Для формирования выходного файла после инференса

In [3]:
def read_rectangle(roi_x, roi_y, roi_width, roi_height):
    x1 = roi_x
    y1 = roi_y
    a = roi_width
    b = roi_height
    vertices = [
        {"x": x1, "y": y1},
        {"x": x1 + a, "y": y1},
        {"x": x1 + a, "y": y1 + b},
        {"x": x1, "y": y1 + b},
        {"x": x1, "y": y1},
    ]
    return vertices

In [4]:
def read_cluster(cluster_class_ind, cluster_type, image_name):
    cluster_classes = {1: "бесформенная структура с упорядоченным расположением клеток",
                    2: "бесформенная структура с неупорядоченным расположением клеток",
                    3: "микрофолликулярная структура",
                    4: "трабекулярная структура",
                    5: "папиллярная структура"
                   }
    cluster = {}
    
    splitted_name = image_name.split('_')
    roi_x = int(float(splitted_name[1].replace('x', '')))
    roi_y = int(float(splitted_name[2].replace('y', '')))
    roi_width = int(float(splitted_name[3].replace('w', '')))
    roi_height = int(float(splitted_name[4].replace('h', '')))
    roi_area = int(float(splitted_name[5].replace('S', '').replace('.png', '')))

    cluster["cluster_class"] = cluster_classes[cluster_class_ind]
    
    rand_probs = np.random.random(5)
    rand_probs = [round(n / rand_probs.sum(), 2) for n in rand_probs]
    rand_probs = sorted(rand_probs, reverse=True)
    cluster["probs"] = rand_probs
    to_change = cluster["probs"][cluster_class_ind - 1]
    cluster["probs"][cluster_class_ind - 1] = cluster["probs"][0]
    cluster["probs"][0] = to_change
    
    if cluster_type == 'usual':
        min_th_cell_num = 10
        max_th_cell_num = 50
    elif cluster_type == 'small':
        min_th_cell_num = 10
        max_th_cell_num = 12
    cluster["th_cell_num"] = random.randint(min_th_cell_num, max_th_cell_num)
    
    cluster["points"] = read_rectangle(roi_x, roi_y, roi_width, roi_height)
    cluster["area"] = roi_area
    
    return cluster

In [5]:
def get_analysis_output(wsi_id, wsi_class_ind, cell_numbers, cluster_paths, cluster_type, json_path):
    data = {}
    
    wsi_classes = {0: "Bethesda 1",
                   1: "Bethesda 2",
                   2: "Bethesda 3",
                   3: "Bethesda 4",
                   4: "Bethesda 5",
                   5: "Bethesda 6"}
    data["wsi_id"] = wsi_id
    data["wsi_class"] = wsi_classes[wsi_class_ind]
    rand_probs = np.random.random(6)
    rand_probs = [round(n / rand_probs.sum(), 2) for n in rand_probs]
    rand_probs = sorted(rand_probs, reverse=True)
    data["probs"] = rand_probs
    to_change = data["probs"][wsi_class_ind]
    data["probs"][wsi_class_ind] = data["probs"][0]
    data["probs"][0] = to_change
    
    cluster_characteristics = {}
    total_cluster_areas = []
    total_clusters = []
    total_th_cell_num = []
    
    cell_characteristics = {}
    cell_characteristics["cellularity"] = 0
    total_cells = []
    total_areas = []
    total_diameters = []
    total_aspect_ratios = []
    total_nuclear_cytoplasmic_ratio = []
    total_circularity = []

    cluster_classes = {1: "ordered_cells_shapeless_cluster_num",
                       2: "disordered_cells_shapeless_cluster_num",
                       3: "microfollicle_num",
                       4: "trabecula_num",
                       5: "papillary_num"
               }

    cluster_characteristics["ordered_cells_shapeless_cluster_num"] = 0
    cluster_characteristics["disordered_cells_shapeless_cluster_num"] = 0
    cluster_characteristics["microfollicle_num"] = 0
    cluster_characteristics["trabecula_num"] = 0
    cluster_characteristics["papillary_num"] = 0
    
    cluster_counter = 0
    for key in cluster_paths:
        if os.path.exists(cluster_paths[key]):
            images = os.listdir(cluster_paths[key])
            for image_name in images:
                cluster = read_cluster(key, cluster_type, image_name)
                cluster["cluster_id"] = cluster_counter
                cluster_counter += 1
                cluster_characteristics[cluster_classes[key]] += 1
                total_clusters.append(cluster)
                total_cluster_areas.append(cluster["area"])
                total_th_cell_num.append(cluster["th_cell_num"])
    
    cluster_characteristics["mean_cluster_area"] = round(statistics.mean(total_cluster_areas), 2)
    cluster_characteristics["mean_th_cell_num_in_clusters"] = round(statistics.mean(total_th_cell_num), 2)
    
    # input_cells_num = 0
    # for key in cell_numbers:
    #     if key != 6:
    #         input_cells_num += cell_numbers[key]
            
    # if input_cells_num < sum(total_th_cell_num):
    #     if cluster_type == 'usual':
    #         min_add_cells = 10
    #         max_add_cells = 50
    #     elif cluster_type == 'small':
    #         min_add_cells = 1
    #         max_add_cells = 3
    #     cell_numbers[1] += sum(total_th_cell_num) - input_cells_num + random.randint(min_add_cells, max_add_cells)
            
    # counter = 0
    # for key in cell_numbers:
    #     if key != 6:
    #         cell_characteristics["cellularity"] += cell_numbers[key]
    #     for i in range(cell_numbers[key]):
    #         cell = create_cell(key)
    #         cell["cell_id"] = counter
    #         counter += 1
    #         total_cells.append(cell)
    #         total_areas.append(cell["area"])
    #         total_diameters.append(cell["min_d"])
    #         total_diameters.append(cell["max_d"])
    #         total_aspect_ratios.append(cell["aspect_ratio"])
    #         total_nuclear_cytoplasmic_ratio.append(cell["nuclear_cytoplasmic_ratio"])
    #         total_circularity.append(cell["circularity"])
                
    # cell_characteristics["th_norm_cell_num"] = cell_numbers[1]
    # cell_characteristics["th_gurtle_cell_num"] = cell_numbers[2]
    # cell_characteristics["th_multiple_nuclei_cell_num"] = cell_numbers[3]
    # cell_characteristics["th_pseudoinclusion_cell_num"] = cell_numbers[4]
    # cell_characteristics["th_groove_cell_num"] = cell_numbers[5]
    # cell_characteristics["lymphocyte_num"] = cell_numbers[6]
    
    # cell_characteristics["mean_th_cell_area"] = round(statistics.mean(total_areas), 2)
    # cell_characteristics["mean_th_cell_diameter"] = round(statistics.mean(total_diameters), 2)
    # cell_characteristics["mean_th_cell_aspect_ratio"] = round(statistics.mean(total_aspect_ratios), 2)
    # cell_characteristics["mean_th_cell_nuclear_cytoplasmic_ratio"] = round(statistics.mean(total_nuclear_cytoplasmic_ratio), 2)
    # cell_characteristics["mean_th_cell_circularity"] = round(statistics.mean(total_circularity), 2)

    data["cluster_characteristics"] = cluster_characteristics
    data["cell_clusters"] = total_clusters
    data["cell_characteristics"] = cell_characteristics
    data["cells"] = total_cells
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)

In [6]:
# задать параметры в соответствии с указанными выше обозначениями
wsi_id = 1
wsi_class_ind = 0 # нумерация с 0
cluster_paths = {1: 'D:/Study/CV_Project2/qupath_projeсts/project3_labelled_wsi/exported_rois/results/Per-1-4879-23/ordered',
                2: 'D:/Study/CV_Project2/qupath_projeсts/project3_labelled_wsi/exported_rois/results/Per-1-4879-23/disordered',
                3: 'D:/Study/CV_Project2/qupath_projeсts/project3_labelled_wsi/exported_rois/results/Per-1-4879-23/microfollicle',
                4: 'D:/Study/CV_Project2/qupath_projeсts/project3_labelled_wsi/exported_rois/results/Per-1-4879-23/trabeculae',
                5: 'D:/Study/CV_Project2/qupath_projeсts/project3_labelled_wsi/exported_rois/results/Per-1-4879-23/papillary',
                  }
cell_numbers = {1: 0,
                2: 0,
                3: 0,
                4: 0,
                5: 0,
                6: 0
               }
cluster_type = 'small' # usual or small
json_path = f'D:\\Study\\CV_Project2\\analysis_output\\examples\\2_labeled\\{wsi_id}_labeled.json'

get_analysis_output(wsi_id, wsi_class_ind, cell_numbers, cluster_paths, cluster_type, json_path)